[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_02_mlp_forward/task_1_perceptron_and_activations.ipynb)

# Week 2 · From one neuron to a small network

**Goal:** understand how inputs become predictions. We will calculate one neuron, process a batch, and connect a **2 → 3 → 1** network. Training starts in week 4.

Use your local notebook or Colab on CPU. No downloads, GPU, Kaggle submission or W&B run are needed. In the Colab URL, replace `fiit-ba` with your GitHub username to open your fork.

Complete the four **TODO** cells and answer the three questions. Run cells in order; supplied checks print `OK` or point to a mismatch. This is practice for the test; there is no hand-in.

*Adapted from the FIIT NSIETE course materials (vgg-fiit/NSIETE_2026).*

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

torch.manual_seed(42)
DTYPE = torch.float64  # use double precision for the checks

def check(name, actual, expected):
    torch.testing.assert_close(actual, expected, atol=1e-10, rtol=0)
    print("OK:", name)

## 1. One neuron

A neuron multiplies each input by a weight, adds the results, then adds a bias:

$$z = w_1x_1 + w_2x_2 + b.$$

For $x=(2,-1)$, $w=(0.5,1)$ and $b=0.5$, we get $z=0.5\cdot2+1\cdot(-1)+0.5=0.5$.
The weights control each input's contribution; the bias shifts the result. Run this worked example:

In [ ]:
x = torch.tensor([2., -1.], dtype=DTYPE)
w = torch.tensor([0.5, 1.], dtype=DTYPE)
b = 0.5
z = (w * x).sum() + b
print("neuron output:", z.item())

## 2. Many samples, one calculation

We store **one sample per column**. For a layer with $n$ inputs, $k$ neurons and a batch of $m$ samples:

| Tensor | Shape |
|---|---|
| Inputs `X` | `(n, m)` |
| Weights `W` | `(k, n)` |
| Bias `b` | `(k, 1)` |
| Output `Z` | `(k, m)` |

The same calculation becomes **`W @ X + b`**. `@` multiplies matrices; `+ b` adds the same bias to every sample (broadcasting).

The supplied `Module` lets us write `layer(X)` to call `layer.forward(X)`. Named layers will help us build a network later.

In [ ]:
class Module:
    def __init__(self):
        self.modules = {}

    def add_module(self, module, name):
        self.modules[name] = module

    def __call__(self, X):
        return self.forward(X)

**TODO 1:** implement `Linear.forward` in one line using `@` and `+`. The constructor is supplied: it creates random weights and zero biases.

In [ ]:
class Linear(Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.randn(out_features, in_features, dtype=DTYPE) * 0.1
        self.b = torch.zeros(out_features, 1, dtype=DTYPE)

    def forward(self, X):
        return ...  # TODO 1: W @ X + b

**Check (given):** use the worked neuron's weights on four samples. The first column is the original sample. PyTorch stores samples in rows, so the reference uses `X.T` and transposes its output back.

In [ ]:
X = torch.tensor([[2., 0., -1., 1.], [-1., 1., 2., 0.]], dtype=DTYPE)
neuron = Linear(2, 1)
neuron.W = w.reshape(1, 2)
neuron.b.fill_(b)
check("batch of four samples", neuron(X), F.linear(X.T, neuron.W, neuron.b[:, 0]).T)
check("first sample", neuron(X)[:, 0], z.reshape(1))
print("batch outputs:", neuron(X))

## 3. Add an activation

An activation changes each neuron's output. **ReLU** keeps positive values and replaces negative values with zero: $a=\max(0,z)$. It lets a network represent non-linear relationships.

**TODO 2:** implement ReLU in one line with `torch.clamp(X, min=0)`. The supplied **Sigmoid** maps values into $(0,1)$; we will use it for the network's final output.

In [ ]:
class ReLU(Module):
    def forward(self, X):
        return ...  # TODO 2: keep positives, replace negatives with zero

class Sigmoid(Module):
    def forward(self, X):
        return 1 / (1 + torch.exp(-X))

In [ ]:
values = torch.tensor([[-2., 0., 2.], [1., -1., 3.]], dtype=DTYPE)
check("ReLU", ReLU()(values), torch.relu(values))
check("Sigmoid", Sigmoid()(values), torch.sigmoid(values))

axis = torch.linspace(-4, 4, 100, dtype=DTYPE)
plt.plot(axis, ReLU()(axis), label="ReLU")
plt.plot(axis, Sigmoid()(axis), label="Sigmoid")
plt.xlabel("input")
plt.ylabel("output")
plt.legend()
plt.show()

## 4. Connect a small network

Our network has **2 inputs → 3 hidden neurons → 1 output**:

`Linear(2, 3) → ReLU → Linear(3, 1) → Sigmoid`

Each layer receives the previous layer's output. **TODO 3:** fill the loop with `A = module(A)`. This is the complete forward pass of a network.

In [ ]:
class Model(Module):
    def forward(self, X):
        A = X
        for module in self.modules.values():
            ...  # TODO 3: pass A through this module
        return A

**TODO 4:** create the two `Linear` layers with the input/output sizes shown above. The connections are supplied in the order the data travels.

In [ ]:
hidden = ...  # TODO 4: Linear with 2 inputs and 3 outputs
output = ...  # TODO 4: Linear with 3 inputs and 1 output

model = Model()
model.add_module(hidden, "hidden")
model.add_module(ReLU(), "relu")
model.add_module(output, "output")
model.add_module(Sigmoid(), "sigmoid")
Y = model(X)
print("predictions:", Y)
print("shape:", tuple(Y.shape))  # (1, 4): one prediction per sample

**Check (given):** compare the complete network with PyTorch, using the same weights, biases and inputs. These predictions use random weights; they have not learned anything yet.

In [ ]:
assert hidden.W.shape == (3, 2) and output.W.shape == (1, 3), "Use the 2 → 3 → 1 architecture"
H_ref = torch.relu(F.linear(X.T, hidden.W, hidden.b[:, 0]))
Y_ref = torch.sigmoid(F.linear(H_ref, output.W, output.b[:, 0])).T
check("complete network", Y, Y_ref)
print("All required code checks passed. Answer the questions below.")

## 5. Three short questions

1. With a batch of **10 samples**, what are the shapes of `X`, the hidden layer's output, and `Y`?

   _Your answer here._

2. What is the purpose of ReLU between the two linear layers?

   _Your answer here._

3. The output is between 0 and 1. Does that mean these random predictions are accurate? What must change for the network to learn?

   _Your answer here._

## Ready for the test?

- Restart the kernel and run all cells: every check should print `OK`.
- Explain the three answers and trace one sample through the network without copying code.
- Keep your notebook for revision. **No notebook, commit link or other submission is required.**

Next week you will calculate gradients for this kind of network; week 4 adds parameter updates.


## Optional: explore further

Extra practice if you have time:

- Change `b` in the worked example and rerun the required cells. How do the neuron's outputs move?
- Count the network's weights and biases using their shapes. Are there any parameters in ReLU or Sigmoid?
- Run the supplied plot below. How do Tanh and LeakyReLU treat negative inputs?

In [ ]:
plt.plot(axis, torch.tanh(axis), label="Tanh")
plt.plot(axis, F.leaky_relu(axis, negative_slope=0.1), label="LeakyReLU")
plt.xlabel("input")
plt.ylabel("output")
plt.legend()
plt.show()